# Role Complexity Gap: Support Operations at Anthropic

Data-backed analysis showing the gap between how Anthropic scopes its **Support Operations Specialist, Content & Knowledge Management** role ($131K–$165K) and the complexity of what they're actually building.

Two Support Ops Specialist roles are currently posted — **Content & Knowledge Management** and **Learning & Education** — both at the same $131K–$165K band. This analysis focuses on the **Content & KM** role, with the **Learning & Ed** role included as a comparison point.

**Core argument:** The Content & KM role demands skills comparable to Anthropic's Content/Training Architect roles ($270K–$365K) — it's not a traditional tech writer or librarian position, but a fast-moving AI-era knowledge systems role that has been underscoped and underpriced.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

from classify import (
    add_classifications, add_usd_salary, TO_USD, SENIORITY_ORDER,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# --- Load current data ---
cur = pd.read_csv("anthropic_salaries.csv", dtype={"job_id": str})
cur = cur.dropna(subset=["salary_min", "salary_max"]).copy()
add_classifications(cur)
add_usd_salary(cur)
cur["mid_usd"] = (cur["min_usd"] + cur["max_usd"]) / 2
print(f"Current roles with salary: {len(cur)}")

# --- Load historical data ---
hist = pd.read_csv("anthropic_salaries_historical.csv", dtype={"job_id": str})
hist = hist.dropna(subset=["salary_min", "salary_max"]).copy()
add_classifications(hist)
add_usd_salary(hist)
hist["mid_usd"] = (hist["min_usd"] + hist["max_usd"]) / 2
hist["first_seen"] = pd.to_datetime(hist["first_seen"], format="%Y%m%d")
hist["last_seen"] = pd.to_datetime(hist["last_seen"], format="%Y%m%d")
hist["quarter"] = hist["first_seen"].dt.to_period("Q")
print(f"Historical roles with salary: {len(hist)}")

In [ ]:
# --- Content family: roles involving writing, editing, content, knowledge, education ---
CONTENT_FAMILY_PATTERNS = [
    # Content + knowledge management
    r"content.*knowledge|knowledge.*management",
    r"certification content|training content",
    r"content (designer|design|developer|engineer|lead|marketing|manager)",
    r"copy and content",
    r"technical video content",
    # Writing / documentation / librarian
    r"technical (documentation|writer|writing)",
    r"prompt engineer.*librarian",
    r"\bcopywriter\b",
    r"\beditorial\b|managing editor",
    # Education (content-adjacent)
    r"developer education|consumer education",
]

def is_content_family(title: str) -> bool:
    if not isinstance(title, str):
        return False
    # Exclude false positives: "Brand Designer, Web Editorial Experience" is design, not writing
    if re.search(r"brand designer", title, re.I):
        return False
    return any(re.search(p, title, re.I) for p in CONTENT_FAMILY_PATTERNS)

cur["content_family"] = cur["title"].apply(is_content_family)
hist["content_family"] = hist["title"].apply(is_content_family)

# --- Support org: all roles with "support" in the title ---
def is_support_role(title: str) -> bool:
    if not isinstance(title, str):
        return False
    t = title.lower()
    return "support" in t and "it support" not in t

cur["support_org"] = cur["title"].apply(is_support_role)
hist["support_org"] = hist["title"].apply(is_support_role)

print(f"Current content family roles: {cur['content_family'].sum()}")
print(f"Current support org roles: {cur['support_org'].sum()}")
print(f"\nHistorical content family roles: {hist['content_family'].sum()}")
print(f"Historical support org roles: {hist['support_org'].sum()}")

# Show current content family
cf = cur[cur["content_family"]][["title", "department", "min_usd", "max_usd", "mid_usd"]].copy()
cf = cf.sort_values("mid_usd", ascending=False)
print("\nCurrent content-family roles:")
for _, r in cf.iterrows():
    print(f"  ${r['min_usd']:>9,.0f} – ${r['max_usd']:>9,.0f}  {r['department'][:30]:<30s}  {r['title']}")

# Show historical content family
cf_h = hist[hist["content_family"]][["title", "first_seen", "min_usd", "max_usd", "mid_usd"]].copy()
cf_h = cf_h.sort_values("first_seen")
print(f"\nHistorical content-family roles ({len(cf_h)} total):")
for _, r in cf_h.iterrows():
    print(f"  {r['first_seen'].strftime('%Y-%m')}  ${r['min_usd']:>9,.0f} – ${r['max_usd']:>9,.0f}  {r['title']}")

## 1. The Salary Gap

Content-adjacent roles at Anthropic span departments but share overlapping skills: content strategy, knowledge management, technical writing, information architecture, and AI tool fluency. The pay gap between them is extreme.

In [ ]:
# --- Content family salary comparison ---
cf = cur[cur["content_family"]].copy()
cf = cf.sort_values("mid_usd", ascending=True)

fig, ax = plt.subplots(figsize=(14, max(4, len(cf) * 0.7)))

# Draw range bars — highlight the Content & KM role in red
TARGET_ID = "5080929008"  # Support Ops Specialist, Content & Knowledge Management
colors = ["#d62728" if str(row["job_id"]) == TARGET_ID else "#4a7cc9"
          for _, row in cf.iterrows()]

y_pos = range(len(cf))
for i, (_, row) in enumerate(cf.iterrows()):
    ax.barh(i, row["max_usd"] - row["min_usd"], left=row["min_usd"],
            color=colors[i], alpha=0.8, height=0.6, edgecolor="white")
    ax.plot(row["mid_usd"], i, "|", color="black", markersize=15, markeredgewidth=2)
    ax.text(row["max_usd"] + 5000, i, f"${row['min_usd']/1000:.0f}K–${row['max_usd']/1000:.0f}K",
            va="center", fontsize=9, fontweight="bold" if str(row["job_id"]) == TARGET_ID else "normal")

ax.set_yticks(list(y_pos))
labels = [t[:60] + "..." if len(t) > 60 else t for t in cf["title"]]
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Annual Salary (USD)")
ax.set_title("Content-Family Roles at Anthropic: Salary Ranges\n(Red = Support Ops Specialist, Content & Knowledge Management)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

# Gap calculation
target_mid = cf.loc[cf["job_id"] == TARGET_ID, "mid_usd"].values[0]
architect_mid = cf.loc[cf["title"].str.contains("Certification Content|Training Content", case=False, regex=True), "mid_usd"].mean()
print(f"\nSupport Ops Specialist (Content & KM) midpoint: ${target_mid:,.0f}")
print(f"Content/Training Architect midpoint: ${architect_mid:,.0f}")
print(f"Gap: ${architect_mid - target_mid:,.0f} ({(architect_mid / target_mid - 1) * 100:.0f}% higher)")

In [ ]:
# --- Content family metadata table ---
cf_table = cur[cur["content_family"]].copy()
cf_table = cf_table.sort_values("mid_usd", ascending=False)

# Extract years of experience from description_md where available
def extract_yoe(desc):
    if not isinstance(desc, str):
        return ""
    m = re.search(r"(\d+)\+?\s*years?", desc)
    return f"{m.group(1)}+" if m else ""

cf_table["yoe"] = cf_table["description_md"].apply(extract_yoe) if "description_md" in cf_table.columns else ""

display_cols = cf_table[["title", "department", "min_usd", "max_usd", "mid_usd", "yoe"]].copy()
display_cols.columns = ["Title", "Department", "Min (USD)", "Max (USD)", "Midpoint", "YoE"]
for col in ["Min (USD)", "Max (USD)", "Midpoint"]:
    display_cols[col] = display_cols[col].apply(lambda x: f"${x:,.0f}")

display_cols.style.set_caption("Content-Family Roles: Salary & Experience Requirements")

In [ ]:
# --- Bubble chart: Experience required vs. salary, bubble size = range width ---
cf_bubble = cur[cur["content_family"]].copy()

# Extract numeric YoE from descriptions
def extract_yoe_num(desc):
    if not isinstance(desc, str):
        return np.nan
    m = re.search(r"(\d+)\+?\s*years?", desc)
    return int(m.group(1)) if m else np.nan

cf_bubble["yoe_num"] = cf_bubble["description_md"].apply(extract_yoe_num)
cf_bubble["range_usd"] = cf_bubble["max_usd"] - cf_bubble["min_usd"]
cf_bubble = cf_bubble.dropna(subset=["yoe_num"])

TARGET_ID = "5080929008"
LEARNING_ID = "5080927008"

fig, ax = plt.subplots(figsize=(14, 8))

# Plot non-target roles first
others = cf_bubble[~cf_bubble["job_id"].isin([TARGET_ID, LEARNING_ID])]
target = cf_bubble[cf_bubble["job_id"] == TARGET_ID]
learning = cf_bubble[cf_bubble["job_id"] == LEARNING_ID]

# Bubble size: scale range to visible sizes
size_scale = 0.003
ax.scatter(others["yoe_num"], others["mid_usd"],
           s=others["range_usd"] * size_scale, alpha=0.6,
           color="#4a7cc9", edgecolors="white", linewidth=1.5,
           label="Other content-family roles", zorder=3)

# Target role — large, red
if not target.empty:
    ax.scatter(target["yoe_num"], target["mid_usd"],
               s=target["range_usd"].values * size_scale * 1.5, alpha=0.9,
               color="#d62728", edgecolors="black", linewidth=2,
               label="Content & KM ($131K–$165K)", zorder=5, marker="D")

# Learning & Ed role — orange diamond
if not learning.empty:
    ax.scatter(learning["yoe_num"], learning["mid_usd"],
               s=learning["range_usd"].values * size_scale * 1.5, alpha=0.9,
               color="#e07b39", edgecolors="black", linewidth=2,
               label="Learning & Ed ($131K–$165K)", zorder=5, marker="D")

# Annotate each bubble
for _, r in cf_bubble.iterrows():
    short = r["title"]
    if len(short) > 42:
        short = short[:39] + "..."
    offset_y = 8 if str(r["job_id"]) not in [TARGET_ID, LEARNING_ID] else -15
    ax.annotate(short, (r["yoe_num"], r["mid_usd"]),
                textcoords="offset points", xytext=(10, offset_y),
                fontsize=7.5, alpha=0.85,
                fontweight="bold" if str(r["job_id"]) in [TARGET_ID, LEARNING_ID] else "normal")

# Reference region showing the gap
ax.axhspan(131_040, 165_000, color="#d62728", alpha=0.07, zorder=0)
ax.axhline(y=200_000, color="gray", linestyle=":", alpha=0.4)
ax.text(cf_bubble["yoe_num"].max() + 0.3, 203_000, "$200K line", fontsize=8, color="gray")

ax.set_xlabel("Required Years of Experience", fontsize=12)
ax.set_ylabel("Salary Midpoint (USD)", fontsize=12)
ax.set_title("Content-Family Roles: Experience vs. Pay\n"
             "Bubble size = salary range width. Both Support Ops Specialists sit far below comparable roles.",
             fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax.set_xlim(left=cf_bubble["yoe_num"].min() - 0.5, right=cf_bubble["yoe_num"].max() + 2)
plt.tight_layout()
plt.show()

In [ ]:
# --- Lollipop: How far each historical content role pays above/below Content & KM ---
cf_hist_lollipop = hist[hist["content_family"]].copy()
# De-duplicate by title (keep highest-paying instance of each title)
cf_hist_lollipop = cf_hist_lollipop.sort_values("mid_usd", ascending=False).drop_duplicates(subset="title")
cf_hist_lollipop = cf_hist_lollipop.sort_values("mid_usd", ascending=True)

# Also add the two current Support Ops Specialist roles if not already in historical
for jid, label in [("5080929008", "Support Ops Specialist, Content & KM"),
                    ("5080927008", "Support Ops Specialist, Learning & Ed")]:
    if jid not in cf_hist_lollipop["job_id"].values:
        row = cur[cur["job_id"] == jid]
        if not row.empty:
            cf_hist_lollipop = pd.concat([cf_hist_lollipop, row], ignore_index=True)

cf_hist_lollipop = cf_hist_lollipop.sort_values("mid_usd", ascending=True)

TARGET_MID = (131_040 + 165_000) / 2
cf_hist_lollipop["gap"] = cf_hist_lollipop["mid_usd"] - TARGET_MID

fig, ax = plt.subplots(figsize=(14, max(6, len(cf_hist_lollipop) * 0.38)))

y_pos = np.arange(len(cf_hist_lollipop))
for i, (_, r) in enumerate(cf_hist_lollipop.iterrows()):
    is_target = str(r["job_id"]) in ["5080929008", "5080927008"]
    color = "#d62728" if r["gap"] <= 0 else "#2ca02c"
    if is_target:
        color = "#d62728"

    # Stem
    ax.hlines(y=i, xmin=0, xmax=r["gap"], color=color, alpha=0.7, linewidth=2)
    # Dot
    ax.scatter(r["gap"], i, color=color, s=80, zorder=5,
               edgecolors="black" if is_target else "white", linewidth=1.5,
               marker="D" if is_target else "o")

    # Label with salary
    side = "left" if r["gap"] < 0 else "right"
    offset = -8000 if r["gap"] < 0 else 8000
    ha = "right" if r["gap"] < 0 else "left"
    ax.text(r["gap"] + offset, i, f"${r['mid_usd']/1000:.0f}K",
            va="center", ha=ha, fontsize=8,
            fontweight="bold" if is_target else "normal", color=color)

# Y labels = role titles
titles = []
for _, r in cf_hist_lollipop.iterrows():
    t = r["title"]
    if len(t) > 55:
        t = t[:52] + "..."
    titles.append(t)
ax.set_yticks(y_pos)
ax.set_yticklabels(titles, fontsize=8)

# Zero line
ax.axvline(x=0, color="#d62728", linewidth=2, linestyle="-", alpha=0.8)
ax.text(2000, len(cf_hist_lollipop) - 0.5,
        f"Content & KM midpoint\n(${TARGET_MID:,.0f})",
        fontsize=9, color="#d62728", fontweight="bold", va="top")

ax.set_xlabel("Salary Gap vs. Content & KM Midpoint (USD)")
ax.set_title("Every Content-Family Role at Anthropic (Historical + Current)\n"
             "Distance from the Content & KM midpoint — nearly all pay significantly more",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"+${x:,.0f}" if x > 0 else f"${x:,.0f}"))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# Stats
above = (cf_hist_lollipop["gap"] > 0).sum()
total = len(cf_hist_lollipop)
print(f"\n{above} of {total} content-family roles ({above/total*100:.0f}%) pay more than Content & KM")
print(f"Median gap: +${cf_hist_lollipop[cf_hist_lollipop['gap'] > 0]['gap'].median():,.0f}")

In [ ]:
# --- Historical timeline: bubbles sized by salary, color by tier ---
cf_hist = hist[hist["content_family"]].copy()
cf_hist = cf_hist.sort_values("first_seen")

def salary_tier(mid):
    if mid < 200_000:
        return "Support tier (<$200K)"
    return "Architect/Lead tier ($200K+)"

cf_hist["tier"] = cf_hist["mid_usd"].apply(salary_tier)
tier_colors = {"Support tier (<$200K)": "#d62728", "Architect/Lead tier ($200K+)": "#4a7cc9"}

fig, ax = plt.subplots(figsize=(16, 8))

# Bubble size proportional to salary midpoint — makes the gap visceral
size_min, size_max = 60, 600
sal_min = cf_hist["mid_usd"].min()
sal_max = cf_hist["mid_usd"].max()

for tier, grp in cf_hist.groupby("tier"):
    sizes = size_min + (grp["mid_usd"] - sal_min) / (sal_max - sal_min) * (size_max - size_min)
    ax.scatter(grp["first_seen"], grp["mid_usd"], s=sizes, alpha=0.7,
               color=tier_colors[tier], label=tier, edgecolors="white",
               linewidth=1.5, zorder=5)
    # Vertical range lines
    for _, r in grp.iterrows():
        ax.plot([r["first_seen"], r["first_seen"]], [r["min_usd"], r["max_usd"]],
                color=tier_colors[tier], alpha=0.3, linewidth=2, zorder=2)

# Annotate roles
for _, r in cf_hist.iterrows():
    short = r["title"]
    if len(short) > 42:
        short = short[:39] + "..."
    ax.annotate(short, (r["first_seen"], r["mid_usd"]),
                textcoords="offset points", xytext=(10, 6),
                fontsize=7, alpha=0.8, rotation=12)

# Reference band at Content & KM salary
ax.axhspan(131_040, 165_000, color="#d62728", alpha=0.08, zorder=0)
ax.axhline(y=165_000, color="#d62728", linestyle="--", alpha=0.5, linewidth=1)
ax.text(cf_hist["first_seen"].min(), 170_000,
        "$165K ceiling (Support Ops Content & KM)",
        color="#d62728", fontsize=9, alpha=0.7, fontweight="bold")

ax.set_title("Historical Content-Family Roles: Salary Over Time\n"
             "Bubble size = salary midpoint. Red shaded band = Content & KM range.",
             fontsize=13, fontweight="bold")
ax.set_xlabel("First Seen")
ax.set_ylabel("Salary Midpoint (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Historical content-family roles: {len(cf_hist)}")
print(f"Below $200K midpoint: {(cf_hist['mid_usd'] < 200_000).sum()}")
print(f"At/above $200K midpoint: {(cf_hist['mid_usd'] >= 200_000).sum()}")
below = cf_hist[cf_hist["mid_usd"] < 200_000]
if len(below) > 0:
    print(f"\nSub-$200K content roles:")
    for _, r in below.iterrows():
        print(f"  ${r['mid_usd']:>9,.0f}  {r['first_seen'].strftime('%Y-%m')}  {r['title']}")

## 2. Support Org Scaling Trajectory

Anthropic's support organization is being built from scratch in real time. The rate of specialization and geographic expansion signals a complex, fast-moving operation — not a steady-state maintenance function.

In [ ]:
# --- Support role postings over time ---
support_hist = hist[hist["support_org"]].copy()
support_by_q = support_hist.groupby("quarter").size().reset_index(name="roles_posted")
support_by_q["quarter_str"] = support_by_q["quarter"].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(support_by_q["quarter_str"], support_by_q["roles_posted"],
              color="#4a7cc9", edgecolor="white")

# Annotate counts
for bar, val in zip(bars, support_by_q["roles_posted"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha="center", fontsize=11, fontweight="bold")

ax.set_title("Support Org: New Role Postings by Quarter", fontsize=13, fontweight="bold")
ax.set_xlabel("Quarter")
ax.set_ylabel("Roles Posted")
ax.set_ylim(top=support_by_q["roles_posted"].max() + 2)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Total support roles posted historically: {len(support_hist)}")
print(f"Currently active: {support_hist['is_active'].sum() if 'is_active' in support_hist.columns else 'N/A'}")

In [ ]:
# --- Title evolution timeline ---
support_hist_sorted = support_hist.sort_values("first_seen")

# Categorize titles into role types
def support_role_type(title):
    t = title.lower() if isinstance(title, str) else ""
    if "head of" in t:
        return "Head / Director"
    if "manager" in t:
        return "Manager"
    if "analyst" in t:
        return "Analyst"
    if "operations specialist" in t or "ops specialist" in t:
        return "Operations Specialist"
    return "Support Specialist"

support_hist_sorted["role_type"] = support_hist_sorted["title"].apply(support_role_type)
type_colors = {
    "Support Specialist": "#4a7cc9",
    "Operations Specialist": "#e07b39",
    "Analyst": "#2ca02c",
    "Manager": "#9467bd",
    "Head / Director": "#d62728",
}

fig, ax = plt.subplots(figsize=(14, max(5, len(support_hist_sorted) * 0.35)))

for i, (_, row) in enumerate(support_hist_sorted.iterrows()):
    color = type_colors.get(row["role_type"], "gray")
    ax.barh(i, (row["last_seen"] - row["first_seen"]).days + 1,
            left=row["first_seen"], color=color, alpha=0.8, height=0.7, edgecolor="white")
    label = row["title"]
    if len(label) > 55:
        label = label[:52] + "..."
    ax.text(row["last_seen"] + pd.Timedelta(days=5), i,
            f"{label}  (${row['mid_usd']/1000:.0f}K)",
            va="center", fontsize=8)

ax.set_yticks(range(len(support_hist_sorted)))
ax.set_yticklabels(["" for _ in range(len(support_hist_sorted))])
ax.set_xlabel("Date")
ax.set_title("Support Org: Role Timeline & Specialization", fontsize=13, fontweight="bold")
ax.invert_yaxis()

# Legend
handles = [mpatches.Patch(color=c, label=l) for l, c in type_colors.items()]
ax.legend(handles=handles, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Geographic expansion ---
support_geo = support_hist.sort_values("first_seen").copy()

def extract_region(loc):
    if not isinstance(loc, str):
        return "Unknown"
    loc_lower = loc.lower()
    if "london" in loc_lower or "uk" in loc_lower:
        return "UK"
    if "dublin" in loc_lower or "ireland" in loc_lower:
        return "Ireland"
    return "US"

support_geo["region"] = support_geo["location"].apply(extract_region)
support_geo["shift_pattern"] = support_geo["title"].apply(
    lambda t: "Non-standard" if isinstance(t, str) and re.search(r"wed.?sun|weekend|night", t, re.I) else "Standard"
)

geo_summary = support_geo.groupby(["quarter", "region"]).size().unstack(fill_value=0)
print("Support roles by quarter and region:")
print(geo_summary.to_string())

print(f"\nRegions over time:")
for q in sorted(support_geo["quarter"].unique()):
    q_roles = support_geo[support_geo["quarter"] == q]
    regions = sorted(q_roles["region"].unique())
    shifts = q_roles["shift_pattern"].value_counts().to_dict()
    print(f"  {q}: {', '.join(regions)} | {len(q_roles)} roles | shifts: {shifts}")

## 3. Skills Overlap Analysis

Comparing what the Support Ops Specialist (Content & KM) job actually requires vs. the Content/Training Architect roles, using the job description text.

In [ ]:
# --- Keyword extraction from job descriptions ---
SKILL_CATEGORIES = {
    "Content Strategy": r"content (strategy|roadmap|gap|audit|prioritiz)|editorial|content.{0,10}(plan|calendar)",
    "Knowledge Management": r"knowledge (management|base|system)|single source of truth|help center|self.?service|information (management|governance)",
    "Information Architecture": r"information architecture|content architecture|modular|taxonomy|navigation|competency framework|curriculum",
    "Technical Writing": r"technical (writing|writer|content|documentation)|complex.{0,15}(concept|topic).{0,15}(accessible|clear)|translate.{0,15}complex|distill.{0,15}complex",
    "Metrics & Data": r"metric|data.?driven|measure.{0,10}(effectiveness|performance|outcome)|analytics|deflection rate|adoption rate",
    "AI / ML Tools": r"\bAI\b|\bLLM|claude|machine learning|AI.?(augment|native|powered|support|tool)|artificial intelligence",
    "Systems Design": r"system.{0,10}(design|build|architect|thinking)|automated|pipeline|tooling|operational (tooling|system)|workflow.{0,10}(build|design|automat)",
    "Cross-functional": r"cross.?functional|collaborate.{0,15}(product|engineering|GTM|partner|stakeholder)|partner.{0,10}with",
    "Program / Project Mgmt": r"program manag|project manag|onboarding program|enablement|curriculum develop",
    "Scale & Growth": r"scale|scaling|growth|expand|from scratch|building.{0,10}(from|ground)|fast.?mov|rapid",
}

# Roles to compare
COMPARE_IDS = {
    "5080929008": "Support Ops Specialist\n(Content & KM)\n$131K–$165K",
    "5080927008": "Support Ops Specialist\n(Learning & Ed)\n$131K–$165K",
    "5097348008": "Certification Content\n& Systems Architect\n$270K–$365K",
    "5097351008": "Training Content\n& Systems Architect\n$270K–$365K",
    "5126762008": "Developer\nEducation Lead\n$290K–$365K",
}

# Build skills matrix from descriptions
skills_matrix = {}
for job_id, label in COMPARE_IDS.items():
    row = cur[cur["job_id"] == job_id]
    if row.empty or "description_md" not in row.columns:
        skills_matrix[label] = {cat: 0 for cat in SKILL_CATEGORIES}
        continue
    desc = row.iloc[0]["description_md"]
    if not isinstance(desc, str):
        desc = ""
    scores = {}
    for cat, pattern in SKILL_CATEGORIES.items():
        matches = re.findall(pattern, desc, re.I)
        scores[cat] = min(len(matches), 3)  # cap at 3 for visual
    skills_matrix[label] = scores

skills_df = pd.DataFrame(skills_matrix)
print("Skills presence (0=absent, 1–3=strength of signal):")
print(skills_df.to_string())

In [ ]:
# --- Skills overlap heatmap ---
fig, ax = plt.subplots(figsize=(14, 7))

# Binary version: present (1) or absent (0) for cleaner visual
skills_binary = (skills_df > 0).astype(int)

sns.heatmap(
    skills_binary, annot=skills_df.values, fmt="d",
    cmap="YlOrRd", linewidths=1, ax=ax,
    cbar_kws={"label": "Signal Strength (0–3)"},
    xticklabels=True, yticklabels=True,
)
ax.set_title("Skills Overlap: Support Ops Specialist vs. Content Architect Roles",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Skill Category")
ax.set_xlabel("")
plt.xticks(rotation=0, ha="center", fontsize=9)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# Overlap calculation
target_col = [c for c in skills_df.columns if "Content & KM" in c][0]
target_skills = set(skills_df.index[skills_df[target_col] > 0])
for col in skills_df.columns:
    if col == target_col:
        continue
    col_skills = set(skills_df.index[skills_df[col] > 0])
    shared = target_skills & col_skills
    pct = len(shared) / max(len(target_skills | col_skills), 1) * 100
    print(f"{col[:40]:<40s} overlap: {pct:.0f}% ({len(shared)} shared skills)")

### The "Systems vs. Execution" Framing Gap

The salary difference maps to **one framing distinction**: the Architect roles are described as *building the systems that govern content*, while the Support Ops roles are described as *executing within those systems*.

But critically, **the "systems" in the Architect titles are content/information systems, not traditional software systems design**. The descriptions make this explicit:
- Certification Content & Systems Architect: *"curriculum architecture, assessment design, competency frameworks, and the AI-native systems that let one educator maintain and evolve certification content"*
- Training Content & Systems Architect: *"designing content architectures that are modular enough for AI-assisted adaptation"* and explicitly states *"this isn't an engineering role, but you should be unintimidated by technical content and comfortable building light tooling or automations"*

These are **content management systems, AI-augmented workflows, and quality assurance loops** — the same fundamental discipline as knowledge management. The gap between "architect content systems for training/certification" and "manage knowledge systems for support content" is a scope/framing difference within the same discipline, not a fundamentally different kind of work.

Meanwhile, the **Content & KM** posting actually includes systems-adjacent work:
- **"Develop and maintain content for the AI support system"** — building AI-powered content, not just writing static articles
- **"Monitor performance metrics across self-service channels to identify content gaps"** — data-driven content strategy
- **"Audit existing content for quality, consistency, and technical accuracy"** — content governance

Similarly, the **Learning & Ed** posting includes:
- **"Design, implement, and continuously improve onboarding programs"** — program design from scratch
- **"Build feedback loops for identifying additional support/training needs"** — systems thinking
- **"Partner on internal knowledge management strategy"** — strategic KM, not just execution

This is especially significant given the org is being **built from scratch**. There are no established systems to execute within — the people in these roles will inevitably be designing and building the systems, not just operating them.

The Architect roles explicitly state: *"one person should produce the output that historically required a full enablement team."* But in a brand-new support org with 0 existing infrastructure, the Support Ops Specialists face the same mandate with less pay and less senior titles.

## 4. The Complexity Argument

Scoring roles on complexity dimensions to show that the Support Ops Specialist (Content & KM) role maps closer to the Architect tier than to traditional support.

### Methodology

Each role's job description is scored on **6 dimensions** via regex pattern matching:

| Dimension | What it captures |
|---|---|
| **AI Integration** | AI/LLM/Claude as core to the role |
| **Systems Scope** | Building systems, tooling, automation, pipelines |
| **Audience Complexity** | Variety of stakeholders (developers, enterprise, partners, etc.) |
| **Rate of Change** | Scaling, greenfield, fast-moving signals |
| **Cross-functional Breadth** | Collaboration across teams (Product, Engineering, GTM, etc.) |
| **Content Governance** | Quality control, auditing, standards, consistency |

Each dimension is scored 0–5 (count of regex matches, capped). Total score ranges 0–30. The "comparable complexity" analysis then finds all current Anthropic roles scoring within ±3 points of the Content & KM role's total.

**Limitations:** This is a keyword-frequency proxy, not semantic analysis. It's most useful as a *relative* comparison across Anthropic's own postings, which follow a consistent description structure.

In [ ]:
# --- Role complexity scoring ---
COMPLEXITY_DIMS = {
    "AI Integration": r"\bAI\b|\bLLM|claude|AI.?(augment|native|powered|support system)|machine learning",
    "Systems Scope": r"system|architect|infrastructure|platform|pipeline|tooling|automat|workflow|single source of truth",
    "Audience Complexity": r"developer|enterprise|partner|trainer|champion|customer|end.?user|stakeholder",
    "Rate of Change": r"fast.?mov|rapid|evolv|scale|from scratch|growing|new.{0,10}(product|feature|launch)|keeping.{0,10}current",
    "Cross-functional Breadth": r"cross.?functional|product|engineering|GTM|partnership|legal|marketing|sales|collaborate.{0,15}with",
    "Content Governance": r"audit|quality|consistency|accuracy|governance|rubric|standard|single source|maintain",
}

# Short, unique legend labels for each role
LEGEND_LABELS = {
    "5080929008": "Content & KM ($131K–$165K)",
    "5080927008": "Learning & Ed ($131K–$165K)",
    "5097348008": "Cert. Architect ($270K–$365K)",
    "5097351008": "Training Architect ($270K–$365K)",
    "5126762008": "Dev Education Lead ($290K–$365K)",
}

complexity = {}
legend_keys = {}
for job_id, label in COMPARE_IDS.items():
    row = cur[cur["job_id"] == job_id]
    desc = row.iloc[0]["description_md"] if not row.empty and "description_md" in row.columns else ""
    if not isinstance(desc, str):
        desc = ""
    scores = {}
    for dim, pattern in COMPLEXITY_DIMS.items():
        matches = re.findall(pattern, desc, re.I)
        scores[dim] = min(len(matches), 5)  # cap at 5
    scores["Total"] = sum(scores.values())
    legend_key = LEGEND_LABELS.get(job_id, label.split("\n")[0])
    complexity[legend_key] = scores
    legend_keys[job_id] = legend_key

comp_df = pd.DataFrame(complexity)

# Grouped bar chart
dims = list(COMPLEXITY_DIMS.keys())
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(dims))
n_roles = len(complexity)
width = 0.8 / n_roles
role_colors = ["#d62728", "#e07b39", "#4a7cc9", "#2ca02c", "#9467bd"]

for i, (legend_label, scores) in enumerate(complexity.items()):
    vals = [scores[d] for d in dims]
    ax.bar(x + i * width, vals, width, label=legend_label, color=role_colors[i], alpha=0.85)

ax.set_xticks(x + width * (n_roles - 1) / 2)
ax.set_xticklabels(dims, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Signal Count (capped at 5)")
ax.set_title("Role Complexity by Dimension", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

# Print totals
print("\nComplexity totals:")
for legend_label, scores in complexity.items():
    print(f"  {legend_label:<42s} score: {scores['Total']}")

In [ ]:
# --- Where does the Content & KM role sit in the overall Anthropic pay landscape? ---
usd_roles = cur[cur["currency"] == "USD"].copy()

# Target: Support Ops Specialist, Content & Knowledge Management
target_min = 131_040
target_max = 165_000
target_mid = (target_min + target_max) / 2

percentile = (usd_roles["mid_usd"] <= target_mid).mean() * 100

fig, ax = plt.subplots(figsize=(14, 5))
sns.histplot(usd_roles["mid_usd"], bins=35, kde=True, ax=ax, color="#4a7cc9", alpha=0.6)

# Highlight target range
ax.axvspan(target_min, target_max, color="#d62728", alpha=0.25,
           label="Content & KM / Learning & Ed range")
ax.axvline(target_mid, color="#d62728", linestyle="--", linewidth=2,
           label=f"Content & KM midpoint: ${target_mid:,.0f}")

ax.set_title(f"Anthropic USD Salary Distribution\n"
             f"Support Ops Specialist (Content & KM) sits at the {percentile:.0f}th percentile",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Annual Salary Midpoint (USD)")
ax.set_ylabel("Number of Roles")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Support Ops Specialist (Content & KM) midpoint: ${target_mid:,.0f}")
print(f"Support Ops Specialist (Learning & Ed) midpoint: ${target_mid:,.0f}  (same band)")
print(f"Anthropic USD median: ${usd_roles['mid_usd'].median():,.0f}")
print(f"Percentile: {percentile:.1f}th")
print(f"Below this midpoint: {(usd_roles['mid_usd'] <= target_mid).sum()} roles")
print(f"Above this midpoint: {(usd_roles['mid_usd'] > target_mid).sum()} roles")

In [ ]:
# --- What comparable complexity pays ---
# Score ALL current roles on the same complexity dimensions used in cell-18,
# then find roles whose total score is close to the Content & KM role.

SUPPORT_OPS_IDS = {"5080929008", "5080927008"}

all_scores = []
for _, row in cur.iterrows():
    desc = row.get("description_md", "")
    if not isinstance(desc, str):
        desc = ""
    total = 0
    for dim, pattern in COMPLEXITY_DIMS.items():
        matches = re.findall(pattern, desc, re.I)
        total += min(len(matches), 5)
    all_scores.append({"job_id": str(row["job_id"]), "title": row["title"],
                        "department": row["department"], "mid_usd": row["mid_usd"],
                        "min_usd": row["min_usd"], "max_usd": row["max_usd"],
                        "currency": row["currency"],
                        "complexity_score": total})

scores_df = pd.DataFrame(all_scores)

# Get the Content & KM complexity score
target_score = scores_df.loc[scores_df["job_id"] == "5080929008", "complexity_score"].values[0]
target_mid = scores_df.loc[scores_df["job_id"] == "5080929008", "mid_usd"].values[0]

# Find roles within ±3 points of the Content & KM score
TOLERANCE = 3
comparable = scores_df[
    (scores_df["complexity_score"] >= target_score - TOLERANCE) &
    (scores_df["complexity_score"] <= target_score + TOLERANCE) &
    (scores_df["currency"] == "USD")  # apples-to-apples
].sort_values("mid_usd", ascending=False)

print(f"Content & KM complexity score: {target_score}")
print(f"Roles with similar complexity (score {target_score - TOLERANCE}–{target_score + TOLERANCE}, USD only): {len(comparable)}")
print("=" * 105)
for _, r in comparable.iterrows():
    marker = " <<<" if r["job_id"] in SUPPORT_OPS_IDS else ""
    print(f"  ${r['mid_usd']:>9,.0f}  (score {r['complexity_score']:>2d})  "
          f"{r['department'][:30]:<30s}  {r['title']}{marker}")

# Stats excluding both Support Ops Specialist roles
others = comparable[~comparable["job_id"].isin(SUPPORT_OPS_IDS)]
if len(others) > 0:
    print(f"\nComparable-complexity roles (excl. Support Ops Specialists):")
    print(f"  Count:  {len(others)}")
    print(f"  Median: ${others['mid_usd'].median():,.0f}")
    print(f"  Min:    ${others['mid_usd'].min():,.0f}")
    print(f"  Max:    ${others['mid_usd'].max():,.0f}")
    print(f"\n  Content & KM midpoint: ${target_mid:,.0f}")
    print(f"  Gap to median:         ${others['mid_usd'].median() - target_mid:,.0f}")
    pct_above = (others["mid_usd"] > target_mid).mean() * 100
    print(f"  % of comparable roles paying more: {pct_above:.0f}%")

## 5. Key Takeaways

### 1. Lowest-paid writing/content role Anthropic has ever posted
Across 20+ historical writing, editorial, content, and documentation roles at Anthropic, both Support Ops Specialists — **Content & KM** and **Learning & Ed** — at $131K–$165K are the lowest-paid by a wide margin. For comparison: Copywriter ($150K–$240K), Technical Writer ($250K–$365K), Editorial roles ($200K–$320K), Content Designer ($240K–$300K), Technical Documentation ($255K–$405K), Prompt Engineer & Librarian ($250K–$375K). Even a contract Technical Writer ($156K–$208K) was paid more.

### 2. Substantial skill overlap with roles paying 2x more
The skills heatmap shows that the **Content & KM** role shares the majority of required competencies with the Content/Training Architect roles ($270K–$365K): content strategy, knowledge management, technical writing, metrics-driven improvement, AI tool fluency, and cross-functional collaboration. The **Learning & Ed** role overlaps similarly on program management, metrics, and knowledge management dimensions.

### 3. The support org is being built from scratch
From 1 role in 2024 Q2 to 8+ specialized roles across 3 countries with non-standard shift patterns in 2026 Q1. This is a greenfield build, not steady-state maintenance. The people in both Support Ops Specialist roles will be *creating* the knowledge and education systems, not just operating existing ones.

### 4. Complexity metrics align with $250K+ positions
When scored on AI integration, systems scope, audience complexity, rate of change, and cross-functional breadth, the **Content & KM** role scores in the same range as Anthropic's Architect-tier positions.

### 5. The "systems vs. execution" distinction doesn't hold
The Architect framing is: *build AI-powered systems that scale content without growing headcount*. But in a brand-new support org with zero existing infrastructure, both Support Ops Specialists will be doing exactly this — designing and building content and education systems from the ground up, using AI tools, with metrics-driven iteration. The framing difference is the title, not the work.